# 03 – Model Design Baseline: K-Means

**Proyek:** Analisis Pola Persebaran Catatan Keanekaragaman Hayati di Indonesia Menggunakan Metode Clustering  
**Model:** K-Means (baseline)  
**Jenis pembelajaran:** Unsupervised Learning

Notebook ini menggunakan hasil **Notebook 02**. Satu sampel mewakili satu grid geografis 0,5° × 0,5° pada satu tahun pengamatan. Model menggunakan 9 fitur numerik yang telah diimputasi dan distandarisasi.

**Catatan:** K-Means hanya dilatih dengan data training. Evaluasi dilakukan pada validation. Data test tidak digunakan pada Minggu 3.

## 1. Teori dan alasan pemilihan

K-Means membagi data menjadi K cluster berdasarkan jarak Euclidean ke centroid terdekat. Model dipilih sebagai baseline karena sederhana, relatif cepat, dan sesuai sebagai pembanding awal untuk fitur numerik yang telah distandarisasi.

- **Hypothesis space:** pembagian data menjadi K kelompok yang masing-masing diwakili centroid pada ruang 9 fitur.
- **Inductive bias:** cluster diasumsikan relatif kompak/sferis; setiap sampel masuk ke satu cluster terdekat (*hard assignment*).
- **Kelebihan:** sederhana, cepat, dan mudah dijelaskan.
- **Keterbatasan:** sensitif terhadap outlier dan kurang cocok untuk cluster dengan bentuk atau kepadatan yang sangat berbeda.

`K=4` adalah parameter awal untuk dibandingkan dengan GMM, bukan jumlah cluster optimal yang sudah dipastikan.

## 2. Import library dan konfigurasi

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

DATA_DIR = Path("/content/hasil_data_preparation_clustering")  # sama dengan Notebook 02
MODEL_DIR = Path("/content/hasil_model_clustering")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
K = 4

## 3. Memuat data Notebook 02

Jalankan Notebook 02 sampai selesai **sebelum** menjalankan bagian ini. Gunakan runtime Colab yang memiliki folder hasil Notebook 02.

In [3]:
file_wajib = ["X_train_scaled.npy", "X_validation_scaled.npy",
              "validation_grid_year.csv", "feature_columns.json"]
if not all((DATA_DIR / f).exists() for f in file_wajib):
    from google.colab import files
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Upload 4 file hasil Notebook 02 yang diminta.")
    for nama, isi in files.upload().items():
        (DATA_DIR / nama).write_bytes(isi)

X_train = np.load(DATA_DIR / "X_train_scaled.npy")
X_val = np.load(DATA_DIR / "X_validation_scaled.npy")
meta_val = pd.read_csv(DATA_DIR / "validation_grid_year.csv")
features = json.loads((DATA_DIR / "feature_columns.json").read_text())

assert X_train.shape[1] == X_val.shape[1] == len(features)
assert len(X_val) == len(meta_val)
print("Training:", X_train.shape, "| Validation:", X_val.shape)
print("Jumlah fitur:", len(features))

Upload 4 file hasil Notebook 02 yang diminta.


Saving validation_grid_year.csv to validation_grid_year.csv
Saving feature_columns.json to feature_columns.json
Saving X_validation_scaled.npy to X_validation_scaled.npy
Saving X_train_scaled.npy to X_train_scaled.npy
Training: (4744, 9) | Validation: (1505, 9)
Jumlah fitur: 9


## 4. Melatih K-Means

Parameter awal: `n_clusters=4`, `n_init=20`, `max_iter=300`, dan `random_state=42`. Model hanya di-fit pada training.

In [4]:
model = KMeans(n_clusters=K, init="k-means++", n_init=20,
               max_iter=300, random_state=42)
model.fit(X_train)
labels = model.predict(X_val)
print("K-Means selesai. Inertia training:", round(model.inertia_, 3))

K-Means selesai. Inertia training: 17123.239


In [8]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

silhouette_kmeans = silhouette_score(X_val, labels)
davies_kmeans = davies_bouldin_score(X_val, labels)
calinski_kmeans = calinski_harabasz_score(X_val, labels)

print("Hasil Validasi K-Means")
print("Silhouette Score:", round(silhouette_kmeans, 4))
print("Davies-Bouldin Index:", round(davies_kmeans, 4))
print("Calinski-Harabasz Score:", round(calinski_kmeans, 4))

Hasil Validasi K-Means
Silhouette Score: 0.5078
Davies-Bouldin Index: 0.529
Calinski-Harabasz Score: 1762.821


## 5. Evaluasi validation

Silhouette dan Calinski-Harabasz lebih baik jika tinggi; Davies-Bouldin lebih baik jika rendah. Metrik ini mengevaluasi struktur cluster, bukan akurasi terhadap label.

In [5]:
n_label = np.unique(labels).size
if 1 < n_label < len(labels):
    hasil = {
        "Model": "K-Means", "K": K,
        "Silhouette": silhouette_score(X_val, labels),
        "Davies-Bouldin": davies_bouldin_score(X_val, labels),
        "Calinski-Harabasz": calinski_harabasz_score(X_val, labels)
    }
else:
    hasil = {"Model": "K-Means", "K": K,
             "Silhouette": np.nan, "Davies-Bouldin": np.nan,
             "Calinski-Harabasz": np.nan}

hasil = pd.DataFrame([hasil])
display(hasil.round(4))

,Model,K,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,K-Means,4,0.5078,0.529,1762.821


## 6. Distribusi cluster dan simpan hasil

In [6]:
validasi = meta_val.copy()
validasi["cluster_kmeans"] = labels
ukuran_cluster = validasi["cluster_kmeans"].value_counts().sort_index()
display(ukuran_cluster.rename("Jumlah grid-tahun").to_frame())

joblib.dump(model, MODEL_DIR / "kmeans_baseline.joblib")
hasil.to_csv(MODEL_DIR / "kmeans_validation_metrics.csv", index=False)
validasi.to_csv(MODEL_DIR / "kmeans_validation_labels.csv", index=False)
print("Hasil disimpan di:", MODEL_DIR)

,Jumlah grid-tahun
cluster_kmeans,
0,148
1,615
2,2
3,740


Hasil disimpan di: /content/hasil_model_clustering


## 7. Ringkasan untuk Bab 6.1

- **Model baseline:** K-Means.
- **Parameter awal:** `n_clusters=4`, `init='k-means++'`, `n_init=20`, `max_iter=300`, `random_state=42`.
- **Data pelatihan:** `X_train_scaled.npy` dari Notebook 02.
- **Evaluasi:** Silhouette, Davies-Bouldin, dan Calinski-Harabasz pada validation set.

Gunakan skor yang muncul setelah notebook dijalankan pada dataset proyek untuk mengisi laporan.